In [ ]:
import numpy as np
import pandas as pd

import spatialdata as sd
import spatialdata_io
import spatialdata_plot
from spatialdata_io.experimental import to_legacy_anndata

import anndata as ad
import scanpy as sc
import squidpy as sq

import matplotlib as mpl
mpl.rc('pdf',fonttype=42)
mpl.rcParams['pdf.use14corefonts'] = True
mpl.rcParams['axes.unicode_minus'] = False
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from glob import glob
from natsort import natsorted
from tqdm import tqdm

# load from spaceranger

In [ ]:
sdata = spatialdata_io.visium_hd('../data/spaceranger/1RR/outs', 
                                 dataset_id='1RR', load_all_images=True, var_names_make_unique=True)


In [ ]:
adata = to_legacy_anndata(sdata, table_name='cell_segmentations', include_images=True, coordinate_system="1RR")

In [ ]:
adata.write_h5ad('../results/visiumHD.h5ad')

# add match-seq barcodes

In [ ]:
read_table = pd.read_table('../data/1RR_visiumHD_MATCH_barcode_table.txt.gz')
read_table

In [ ]:
read_table['label'] = read_table['sgRNA'] + read_table['sgRNA_UMI']
read_table['tumor'] = read_table['cell'].map(adata.obs['tumor'])
read_table['label_tumor'] = read_table['label'] + ':' + read_table['tumor'].astype(str)

In [ ]:
filtered_read_table = read_table[read_table['counts'] > 1].copy()
filtered_read_table.shape

In [ ]:
cell_bcs = adata.obs.index
label_bcs = np.array(natsorted(filtered_read_table['label'].unique()))

In [ ]:
cell_map = {k:i for i,k in enumerate(cell_bcs)}
label_map = {k:i for i,k in enumerate(label_bcs)}

In [ ]:
filtered_read_table['cell_coord'] = filtered_read_table['cell'].map(cell_map)
filtered_read_table['label_coord'] = filtered_read_table['label'].map(label_map)

In [ ]:
cell_coord, label_coord = filtered_read_table[['cell_coord', 'label_coord']].dropna().astype(int).values.T

coo = sparse.coo_matrix((np.ones(len(cell_coord)), (cell_coord, label_coord)), dtype=np.int32)
adj = coo.tocsr()
adj

In [ ]:
adata.obsm['labels'] = adj.copy()
adata.uns['labels'] = label_bcs.copy()

In [ ]:
adata.write_h5ad('../results/visiumHD.h5ad')

# cell2location
run with NVIDIA L4 GPU on Google Cloud instance

In [ ]:
import os

import torch

import cell2location
from cell2location.utils.filtering import filter_genes

from copy import deepcopy

In [ ]:
torch.set_float32_matmul_precision('high')

In [ ]:
celltype_key = 'major_celltype'
results_folder = '../results/cell2loc_results/batched'

# create paths and names to results folders for reference regression and cell2location models
ref_run_name = f'{results_folder}/reference_signatures'
run_name = f'{results_folder}/cell2location_map'

## prepare reference

In [ ]:
reference = sc.read_h5ad('../results/adata_proc_singlets.h5ad')
reference

In [ ]:
selected_genes = filter_genes(reference, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)

In [ ]:
# hard filter
reference = reference[:, selected_genes].copy()
reference.shape

In [ ]:
# prepare anndata for the regression model
cell2location.models.RegressionModel.setup_anndata(
    adata=reference,
    batch_key='GEM',
    labels_key=celltype_key,
)

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(reference)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

In [ ]:
mod.train(max_epochs=250)

In [ ]:
# check loss
mod.plot_history(10)

In [ ]:
# In this section, we export the estimated cell abundance (summary of the posterior distribution)
reference = mod.export_posterior(
    reference, sample_kwargs={'num_samples': 1000, 'batch_size': 2500}
)

# Save model
mod.save(f"{ref_run_name}", overwrite=True)

# Save anndata object with results
adata_file = f"{ref_run_name}/sc.h5ad"
reference.write(adata_file)
adata_file

In [ ]:
# load
adata_file = f"{ref_run_name}/sc.h5ad"
reference = sc.read_h5ad(adata_file)
mod = cell2location.models.RegressionModel.load(f"{ref_run_name}", reference)

In [ ]:
# export estimated expression in each cluster
if 'means_per_cluster_mu_fg' in reference.varm.keys():
    inf_aver = reference.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                    for i in reference.uns['mod']['factor_names']]].copy()
else:
    inf_aver = reference.var[[f'means_per_cluster_mu_fg_{i}'
                                    for i in reference.uns['mod']['factor_names']]].copy()
inf_aver.columns = reference.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:5]

## prepare query

In [ ]:
query = sc.read_h5ad('../results/visiumHD.h5ad')
query

In [ ]:
# find shared genes and subset both anndata and reference signatures
intersect = np.intersect1d(query.var_names, inf_aver.index)
query = query[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()

# # prepare anndata for cell2location model - do this later in batch
# cell2location.models.Cell2location.setup_anndata(adata=query, batch_key="batch")
len(intersect)

## random batch training

In [ ]:
# for every "sample", sample with replacement from chunks to allocate 
# some locations from each batch to all training batches
chunk_size = 35_000
chunks = [i for i in range(int(np.ceil(query.n_obs / chunk_size)))]
np.random.seed(0)

query.obs['training_batch'] = 0
for sample in query.obs['batch'].unique():
    ind = query.obs['batch'].isin([sample])
    query.obs.loc[ind, 'training_batch'] = np.random.choice(
        chunks, size=ind.sum(), replace=True, p=None
    )
    
query_full = query.copy()
for k in ['means', 'stds', 'q05', 'q95']:
    query_full.obsm[f"{k}_cell_abundance_w_sf"] = np.zeros((query_full.n_obs, inf_aver.shape[1]))
    
query.obs['training_batch'].value_counts()

In [ ]:
import pyro
import scvi
import gc

In [ ]:
seed = 0
scvi.settings.seed = seed
np.random.seed(seed)

max_epochs = 10000

# submit this chunk as separate jobs
for batch in query.obs['training_batch'].unique():
    # create and train the model
    scvi_run_name = f'{run_name}_batch{batch}_seed{seed}'
    print(scvi_run_name)

    if os.path.exists(scvi_run_name):
        print('output exists, reading and resuming training')
        adata_file = f"{scvi_run_name}/sp.h5ad"
        query = sc.read_h5ad(adata_file)

        training_batch_index = query_full.obs.index.isin(query.obs.index.tolist())

        # load model
        mod = cell2location.models.Cell2location.load(f"{scvi_run_name}", query)

        # write to new location
        i = 1
        new_scvi_run_name = scvi_run_name
        while os.path.exists(new_scvi_run_name):
            new_scvi_run_name = scvi_run_name + f'_{i}'
            i += 1
        scvi_run_name = new_scvi_run_name
        print(f'saving outputs to {new_scvi_run_name}')
        
    else:
        print('starting new training')
        training_batch_index = query_full.obs['training_batch'].isin([batch])
        query = query_full[training_batch_index, :].copy()
    
        # prepare anndata for cell2location model
        cell2location.models.Cell2location.setup_anndata(adata=query, batch_key="batch")
        
        # create and train the model
        mod = cell2location.models.Cell2location(
            query, 
            cell_state_df=inf_aver,
            # the expected average cell abundance: tissue-dependent
            # hyper-prior which can be estimated from paired histology:
            N_cells_per_location=1,
            # hyperparameter controlling normalisation of
            # within-experiment variation in RNA detection:
            detection_alpha=20
        )
        # mod.view_anndata_setup()
        
    # train as normal
    mod.train(max_epochs=max_epochs,
              # if using batch_size=None, train using full data. not recommended to use minibatch (performance loss)
              batch_size=None,
              # use all data points in training because
              # we need to estimate cell abundance at all locations
              train_size=1
             )
    # save model
    mod.save(f"{scvi_run_name}", overwrite=True)
   
    # export posterior
    import pyro
    # In this section, we export the estimated cell abundance (summary of the posterior distribution).
    query = mod.export_posterior(
        query, sample_kwargs={
            'batch_size': int(np.ceil(query.n_obs / 4)), 
            'accelerator': 'gpu',
        },
        add_to_obsm=['q05', 'q95'],
        use_quantiles=True,
    )
    adata_file = f"{scvi_run_name}/sp.h5ad"
    query.write(adata_file)

    # copy cell2location results to the main object
    for k in query_full.obsm.keys():
        query_full.obsm[k][training_batch_index, :] = query.obsm[k].copy()
    query_full.uns[f'mod_{batch}'] = query.uns['mod'].copy()

    del mod
    gc.collect()
    torch.cuda.empty_cache()

os.makedirs(run_name, exist_ok=True)
adata_file = f"{run_name}/sp.h5ad"
query_full.write(adata_file)
print(adata_file)

In [ ]:
# load model for one batch to check loss
mod = cell2location.models.Cell2location.load(f"{scvi_run_name}", query)

In [ ]:
mod.plot_history(10)

In [ ]:
# add 5% quantile, representing confident cell abundance, 'at least this amount is present', to adata.obs with nice names for plotting
query_full.obs[query_full.uns['mod_0']['factor_names']] = query_full.obsm['q05_cell_abundance_w_sf']



In [ ]:
slide = '1RR'

with mpl.rc_context({'axes.facecolor':  'black',
                     'figure.figsize': [5, 5]}):
    
    sc.pl.spatial(local, cmap='magma',
                  color=['cancer', 'CAF', 'macrophage', 'monocyte', 
                         'cDC1', 'cDC2', 'MoDC', 'neutrophil',
                         'CD4', 'CD8', 'Treg', 'NK',
                         'ILC1', 'gdTcell', 'Bcell', 'endothelial'],
                  ncols=4, size=1.3,
                  library_id=f'{slide}_hires_image',
                  # limit color scale at 99.2% quantile of cell abundance
                  vmin=0, vmax='p99.2',
                  save=f'_{slide}_{celltype_key}.png'
                 )

In [ ]:
adata_file = f"{run_name}/sp.h5ad"
query_full.write(adata_file)
print(adata_file)